In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import re
from datasets import load_dataset
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
from sklearn.feature_extraction.text import CountVectorizer
#from cleantext import clean
#import pytorch_lightning as pl
from datasets import Dataset as dts
#from sentence_transformers import SentenceTransformer
from torch.nn.utils.rnn import pack_padded_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [2]:
import numpy as np

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [77]:
dataset = load_dataset('astromis/presuicidal_signals')


In [78]:
ds_train = dataset['train'].to_pandas()
ds_test = dataset['test'].to_pandas()

In [6]:
import emoji

In [ ]:
def emoji_to_clean_russian(text):
    clean_tags = re.sub(r'<emoji>\s*(.*?)\s*</emoji>', lambda m: f":{m.group(1).strip().replace(' ', '_').lower()}:", text)
    emojified = emoji.emojize(clean_tags, language='en')
    

    ru_demojized = emoji.demojize(emojified, language='ru')
    

    final_text = ru_demojized.replace('::', ' ').replace(':', ' ').replace('_', ' ')
    final_text = re.sub(r'\s+', ' ', final_text).strip()
    final_text = re.sub(r'[,;:«»""()\[\]{}]', ' ', final_text)
    return final_text

In [8]:
ds_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 22787 entries, 0 to 22786
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    22787 non-null  str  
 1   label   22787 non-null  int64
dtypes: int64(1), str(1)
memory usage: 3.9 MB


In [9]:
print(f'MEAN LABEL ON TRAIN: {ds_train['label'].mean()}')
print(f'MEAN LABEL ON TEST: {ds_test['label'].mean()}')

MEAN LABEL ON TRAIN: 0.15017334445078334
MEAN LABEL ON TEST: 0.15542131667861164


In [10]:
print(ds_train.isna().sum())
print(ds_test.isna().sum())

text     0
label    0
dtype: int64
text     0
label    0
dtype: int64


In [11]:
def cleaning_text(text):
    text = re.sub(r"</?emoji>", " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [12]:
ds_test['text'] = ds_test['text'].apply(emoji_to_clean_russian)
ds_train['text'] = ds_train['text'].apply(emoji_to_clean_russian)

In [13]:
ds_train['text_len'] = ds_train['text'].str.len()
ds_test['text_len'] = ds_test['text'].str.len()

In [14]:
print(ds_train['text_len'].mean())
print(ds_train['text_len'].min())
print(ds_train['text_len'].max())

90.8253828937552
0
2582


In [15]:
ds_train['count_words'] = ds_train['text'].str.split().str.len()
ds_test['count_words'] = ds_test['text'].str.split().str.len()

In [16]:
print(ds_train['count_words'].mean())
print(ds_train['count_words'].min())
print(ds_train['count_words'].max())

15.44753587571861
0
421


In [17]:
print(len(ds_train[ds_train['count_words'] == 2]))
print(len(ds_test[ds_test['count_words'] == 0]))

923
9


In [18]:
ds_train[ds_train['count_words'] ==2]

,text,label,text_len,count_words
6,слава кпсс!!,0,12,2
12,Давайте поболтаем,0,17,2
18,Есть желающие?#закос,0,20,2
21,люблю герлловвввер,0,18,2
41,Я ЕБАОА,0,7,2
...,...,...,...,...
22592,изгибы тельцА,0,13,2
22596,паучье молоко,0,13,2
22672,я живая,0,7,2
22711,хочу каре...,0,12,2


In [19]:
ds_train['text'][333]

'бог подарил мне подарок в честь дня рождения стэй в лице человека который не смог пить пакетированный чай и сам взял заварил вкуснейший чай бля слезы рекой слезы рекой чел религия религия'

In [20]:
ds_train = ds_train[ds_train['count_words'] >=2]
len(ds_train)

22092

In [21]:
y = ds_train['label']
X = ds_train.drop(columns=['label'])


In [79]:
X_train, X_valid, y_train, y_valid = train_test_split(X,  y, test_size=0.1, stratify=y, random_state=42)

In [23]:
X_train.head()


,text,text_len,count_words
19589,давайте не будем осуждать человека за вкусы в ...,54,9
22025,какое-нибудь аестетик видик сделаю в питере,43,6
14180,if I could fly я бы давно съ€балась куда-нибудь,47,9
18380,у кого-то глебы никиты егоры являются одним ...,90,17
22433,зачем читать и не отвечать,26,5


In [25]:
!uv pip install mawo-razdel emoji

Audited 2 packages in 12ms


In [26]:
from mawo_razdel import tokenize
from collections import Counter

In [103]:
from collections import Counter
import numpy as np


class RNNTokenizer:
    
    def __init__(self, tokenize_fn=None, min_freq=2, max_len=None, lowercase=True):
        self.tokenize_fn = tokenize_fn or (lambda s: s.split())
        self.min_freq = min_freq
        self.max_len = max_len
        self.lowercase = lowercase

        self.vocab = {'<PAD>': 0, '<UNK>': 1}
        self.pad_token_id = 0
        self.unk_token_id = 1

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self._tokenize(text))
        for token, freq in counter.most_common():
            if freq >= self.min_freq:
                self.vocab[token] = len(self.vocab)
        return self

    def fit_max_len(self, texts, pct=99):
        lengths = np.array([len(self._tokenize(t)) for t in texts])
        self.max_len = int(np.percentile(lengths, pct))
        return self.max_len, lengths.mean(), np.percentile(lengths, pct)

    def encode(self, text):
        output = {'input_ids': []}
        for t in text:
            tokens = self._tokenize(t)
            ids = [self.vocab.get(t, self.unk_token_id) for t in tokens]
            if self.max_len is not None:
                ids = ids[:self.max_len]
            output.append(ids)   
        return output

    def __call__(self, text):
        return self.encode(text)

    def _tokenize(self, text):
        tokens = self.tokenize_fn(text)
        return [t.text.lower() for t in tokens] if self.lowercase else list(tokens)

    def __len__(self):
        return len(self.vocab)

In [104]:
tokenizer_rnn = RNNTokenizer(tokenize_fn=tokenize)

In [105]:
tokenizer_rnn.fit_max_len(X_train['text'])
tokenizer_rnn.fit(X_train['text'])

In [102]:
vocab_size = len(tokenizer_rnn.vocab)
vocab_size

17276

In [ ]:
rnn_ds = EncoderDataset(input=X_train)

In [31]:
from torch.nn.utils.rnn import pad_sequence

In [32]:
class SuicideDataset(Dataset):
    def __init__(self, input_ids, labels):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            'ids' : torch.tensor(self.input_ids[index], dtype=torch.long),
            'labels' : torch.tensor(self.labels[index], dtype=torch.float)
            
        }
    

In [33]:
def collate_fn(batch, pad_value):
    padding_value = pad_value

    sequences = [item["ids"] for item in batch]
    labels    = [item["labels"]    for item in batch]


    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=padding_value)
    labels = torch.stack(labels)
    return {
        'ids' : padded_sequences, 'labels': labels
    }
    


In [34]:
train_dataset = SuicideDataset(X_train['input_ids'].tolist(), y_train.to_list())
val_dataset = SuicideDataset(X_valid['input_ids'].tolist(), y_valid.to_list())

In [35]:
train_dataloader = DataLoader(train_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)
val_dataloader = DataLoader(val_dataset, collate_fn=collate_fn, batch_size=4, shuffle=True)

In [38]:
class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_index=0):
        super().__init__()

        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            padding_idx=0,
            embedding_dim=embed_dim
        )
        self.lstm = nn.LSTM(
            input_size=embed_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=True,
                dropout=dropout if num_layers >1 else 0.0
        )
        self.dropout = nn.Dropout(p=dropout)
        self.fc = nn.Linear(hidden_dim* 2, 1)
        self.pad_index = pad_index

    def forward(self, input_ids):

        lengths = (input_ids != self.pad_index).sum(dim=-1)
        embeds = self.embeddings(input_ids) # -> [batch_size, seq_len, embed_dim]

        packed = pack_padded_sequence(embeds, lengths.cpu(), batch_first=True, enforce_sorted=False)


        _, (h_n, _) = self.lstm(packed)

        h = torch.cat([h_n[-2], h_n[-1]],dim=-1)

        logits = self.fc(self.dropout(h))

        return logits.squeeze(-1)




In [40]:
model = RNNClassifier(vocab_size=vocab_size, embed_dim=64, hidden_dim=128, num_layers=1, dropout=0.3).to(device)



In [ ]:
from sklearn.metrics import recall_score, auc, f1_score, precision_recall_curve
from sklearn.metrics import roc_auc_score
import gc

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]




gc.collect()

In [46]:
pos_weight = torch.tensor([((1 - y_train.mean()) / y_train.mean())], device=device)

In [ ]:
num_epochs=5

In [ ]:
#pos_weight = torch.tensor([((1- y_train.mean())/ y_train.mean())/2], device=device) 
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)




pbar = tqdm(total=len(train_dataloader)*num_epochs)
for epoch in range(num_epochs):

    model.train()
    train_loss = 0.0

        

        
    for batch in train_dataloader:
        ids = batch['ids'].to(device)
        labels = batch['labels'].to(device)
        optimizer.zero_grad()

        logits = model(ids)

        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        train_loss += loss.item()

        pbar.update(1)

        


    train_loss/=len(train_dataloader)

    print(f'loss after {epoch} epoch: {train_loss}')

    model.eval()


    all_preds = {
        0.1: [],
        0.2: [],
        0.5: [],
        0.3: [],
        0.4: []
    }

    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch in val_dataloader:
            ids = batch['ids'].to(device)
            labels = batch['labels'].to(device)

            logits = model(ids)

            probabilities = torch.sigmoid(logits)

            all_probs.append(probabilities.cpu().numpy())
            for t in thresholds:
                preds = (probabilities >=t).long()
                all_preds[t].append(preds.cpu().numpy())

            all_labels.append(labels.cpu().numpy())

    all_probs = np.concatenate(all_probs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    roc_auc = roc_auc_score(y_true=all_labels, y_score=all_probs)
    



    positive_probs = all_probs[all_labels == 1]
    negative_probs = all_probs[all_labels == 0]


    for key in all_preds:

        all_preds[key] = np.concatenate(all_preds[key], axis=0)

        print(f'Threshold - {key}, f1: {f1_score(y_pred=all_preds[key], y_true=all_labels)}' 
            
            f' recall: {recall_score(y_pred=all_preds[key], y_true=all_labels)}'
            f' roc-auc: {roc_auc}')

    print(f' \n EPOCH: {epoch} \n MEAN POS PROBS: {positive_probs.mean()} \n MEAN NEG POS PROBS: {negative_probs.mean()} '  )
                    


pbar.close()



После нескольких тестовых запусков, видно, что после 1 эпохи метрики начинают падать, так же, как и  вероятности для положительного класса, зная, что целевого класса 15%, можно попробовать сделать оверсемплинг.

In [48]:
from torch.utils.data import WeightedRandomSampler

In [ ]:
class_counts = np.bincount(y_train)
class_weight = 1.0/ class_counts
sample_weights = class_weight[y_train]
sampler = WeightedRandomSampler(
weights=torch.from_numpy(sample_weights).double(), num_samples=len(sample_weights),
replacement=True
)

In [51]:
oversampling_train_dataloader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn, sampler=sampler)

In [52]:
from tqdm.auto import tqdm

Лучший roc-auc был 0.64, попробуем использовать энкодер модель.

In [55]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

In [84]:
class EncoderDataset(Dataset):

    

    def __init__(self, input, tokenizer, labels):

        tokenized_text = tokenizer(input.tolist(), padding=False, truncation=True, max_length=128, return_tensors=None)

        
        self.input_ids = tokenized_text['input_ids']
        self.labels = labels.tolist()
        self.attention_mask = tokenized_text['attention_mask']

        
        

    def __len__(self):
        return len(self.labels)


    def __getitem__(self, index):
        return {
            'input_ids': torch.tensor(self.input_ids[index], dtype=torch.long),
            'labels': torch.tensor(self.labels[index], dtype=torch.long),
            'attention_mask': torch.tensor(self.attention_mask[index],dtype=torch.long)

        }

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "clapAI/modernBERT-base-multilingual-sentiment"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id, torch_dtype=torch.float16, num_labels=1, ignore_mismatched_sizes=True, problem_type='multi_label_classification')
pad_token_id = tokenizer.pad_token_id
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [85]:
t_d = EncoderDataset(X_train['text'], tokenizer=tokenizer, labels=y_train)

In [88]:
encoder_train_ds = EncoderDataset(X_train['text'], labels=y_train, tokenizer=tokenizer)
encoder_valid_ds = EncoderDataset(X_train['text'], labels=y_valid, tokenizer=tokenizer)

In [89]:
encoder_train_loader = DataLoader(encoder_train_ds, collate_fn=data_collator, batch_size=4, sampler=sampler)
encoder_valid_loader = DataLoader(encoder_valid_ds, collate_fn=data_collator, batch_size=4)

In [64]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8, weight_decay=0.01)

In [65]:
scheduler = torch.optim.lr_scheduler.LinearLR(optimizer,start_factor=1.0, total_iters=len(encoder_train_loader)*3)

In [66]:
from tqdm.auto import tqdm

In [67]:
epochs = 5

In [68]:
from sklearn.metrics import f1_score, precision_score, accuracy_score

In [ ]:
torch.nn.BCEWithLogitsLoss

In [70]:
best_roc_auc = 0.0

In [108]:
for param in model.parameters():
    param.requires_grad = False

In [ ]:
for param in model.head.parameters():


ModernBertPredictionHead(
  (dense): Linear(in_features=768, out_features=768, bias=False)
  (act): GELUActivation()
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)

In [ ]:
model.to(device)
progress_bar = tqdm(range(len(encoder_train_loader)*epochs))
for epoch in range(epochs):
    model.train()

    total_loss = 0.0


    for batch in encoder_train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        labels = batch['labels'].float().unsqueeze(1)

        outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=labels)


        loss = outputs.loss
        total_loss+=loss.item()
        
        assert torch.isfinite(outputs.logits).all(), 'Nan logits'

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        progress_bar.update(1)

    total_loss/=len(encoder_train_ds)
    print(f'loss after {epoch} epoch: {total_loss}')


    model.eval()
    all_preds, all_labels = [], []
    all_probs = []
    with torch.no_grad():
        for batch in encoder_valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'])

            logits = outputs.logits

            probs = torch.sigmoid(logits)

            all_probs.append(probs.float().squeeze(-1).cpu())

            all_labels.append(batch['labels'].cpu())

    

    
    all_labels = torch.cat(all_labels).numpy()
    all_probs = torch.cat(all_probs).numpy()
    roc_auc = roc_auc_score(y_true=all_labels, y_score=all_probs)

    #positive_probs = all_probs[all_labels == 1]
    #negative_probs = all_probs[all_labels == 0]
    
    
    for t in thresholds:

        preds = (all_probs >= t).astype(int)

        precision = precision_score(all_labels, preds, zero_division=0)
        recall = recall_score(all_labels, preds, zero_division=0)
        f1 = f1_score(all_labels, preds, zero_division=0)
        acc = accuracy_score(all_labels, preds)
            

        print(
            f"Threshold: {t:.1f} | "
            f"Precision: {precision:.3f} | "
            f"Recall: {recall:.3f} | "
            f"F1: {f1:.3f}"
            )
        
    if roc_auc > best_roc_auc:
        best_roc_auc = roc_auc
        best_model_state = model.state_dict().copy()
        print(f"Best model updated (ROC-AUC: {best_roc_auc:.4f})")

    print(f'roc-auc: {roc_auc}, accuracy: {acc}')
                
    
        
    
        



  0%|          | 0/24855 [00:00<?, ?it/s]

/home/ydlawq/torch_rocm_v/.venv/lib/python3.12/site-packages/transformers/integrations/sdpa_attention.py:92: UserWarning: Mem Efficient attention on Current AMD GPU is still experimental. Enable it with TORCH_ROCM_AOTRITON_ENABLE_EXPERIMENTAL=1. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/hip/sdp_utils.cpp:338.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


loss after 0 epoch: 0.0
Threshold: 0.1 | Precision: 0.154 | Recall: 1.000 | F1: 0.267
Threshold: 0.2 | Precision: 0.153 | Recall: 0.988 | F1: 0.265
Threshold: 0.3 | Precision: 0.149 | Recall: 0.924 | F1: 0.256
Threshold: 0.4 | Precision: 0.145 | Recall: 0.800 | F1: 0.246
Threshold: 0.5 | Precision: 0.145 | Recall: 0.618 | F1: 0.235
roc-auc: 0.44962173639509273, accuracy: 0.38190045248868776
loss after 1 epoch: 0.0
Threshold: 0.1 | Precision: 0.154 | Recall: 1.000 | F1: 0.267
Threshold: 0.2 | Precision: 0.153 | Recall: 0.988 | F1: 0.265
Threshold: 0.3 | Precision: 0.149 | Recall: 0.924 | F1: 0.256
Threshold: 0.4 | Precision: 0.145 | Recall: 0.800 | F1: 0.246
Threshold: 0.5 | Precision: 0.145 | Recall: 0.618 | F1: 0.235
roc-auc: 0.44962173639509273, accuracy: 0.38190045248868776


KeyboardInterrupt: 

In [173]:
model.to(device)

ModernBertForSequenceClassification(
  (model): ModernBertModel(
    (embeddings): ModernBertEmbeddings(
      (tok_embeddings): Embedding(50368, 768, padding_idx=50283)
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (drop): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0): ModernBertEncoderLayer(
        (attn_norm): Identity()
        (attn): ModernBertAttention(
          (Wqkv): Linear(in_features=768, out_features=2304, bias=False)
          (Wo): Linear(in_features=768, out_features=768, bias=False)
          (out_drop): Identity()
        )
        (mlp_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): ModernBertMLP(
          (Wi): Linear(in_features=768, out_features=2304, bias=False)
          (act): GELUActivation()
          (drop): Dropout(p=0.0, inplace=False)
          (Wo): Linear(in_features=1152, out_features=768, bias=False)
        )
      )
      (1-21): 21 x ModernBertEncoderLayer(
       

In [177]:
batch = next(iter(encoder_train_loader))
batch = {k: v.to(device) for k, v in batch.items()}
labels = batch['labels'].float().unsqueeze(1)

outputs = model(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], labels=labels)

loss = outputs.loss
loss.backward()

In [194]:
del model